# Tools with LangGraph

In the RAG notebook we used `client.responses.create` and let the **server** handle the agent loop — tool discovery, execution, and feeding results back to the model all happened inside one API call.

Here we move that loop to the **client** using **LangGraph**. The model and the ticketing MCP server are still the same, but now we control each step: we can inspect intermediate state, add human-in-the-loop approvals, or branch the workflow based on a tool result.

Because OGX / Llama Stack exposes an OpenAI-compatible `/v1` endpoint, connecting LangGraph to it requires only one line to change compared to a standard OpenAI setup.

## Install dependencies

In [ ]:
import os
os.environ["PIP_INDEX_URL"] = "https://pypi.org/simple"

# mcp 2.0 changed its API — langchain-mcp-adapters needs mcp 1.x.
# Force-reinstall cleans up any mixed 1.x/2.x files from previous attempts.
!pip install -q --force-reinstall "mcp>=1.6.0,<2.0.0"
!pip install -q langchain-openai langgraph langchain-mcp-adapters mlflow

print("Done. Restart the kernel before running the next cells.")

In [ ]:
import asyncio
import os
import mlflow
import warnings
warnings.filterwarnings("ignore")

from langchain_openai import ChatOpenAI
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent

## Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MLflow
# ─────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI")
MLFLOW_TRACKING_TOKEN = ""
EXPERIMENT_NAME = "hospital-helpdesk"
PROMPT_NAME = "ai-hospital-helpdesk"

os.environ["MLFLOW_TRACKING_TOKEN"] = MLFLOW_TRACKING_TOKEN
os.environ["MLFLOW_WORKSPACE"] = "hospital-helpdesk"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    mlflow.create_experiment(EXPERIMENT_NAME)
mlflow.set_experiment(EXPERIMENT_NAME)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Model — pointing LangChain at Llama Stack / OGX via /v1
# ─────────────────────────────────────────────────────────────────
LLAMA_STACK_URL = os.getenv("LLAMA_STACK_URL", "http://lsd-genai-playground-service.hospital-helpdesk.svc.cluster.local:8321")
LLAMA_STACK_MODEL = os.getenv("LLAMA_STACK_MODEL", "tinyllamatinyllama-11b-chat-v1")

# Internal cluster URL of the MCP ticketing server (streamable-HTTP transport)
MCP_TICKETING_URL = "http://mcp-ticketing.hospital-helpdesk.svc.cluster.local:8080/mcp"

model = ChatOpenAI(
    base_url=f"{LLAMA_STACK_URL}/v1",
    model=LLAMA_STACK_MODEL,
    api_key="not-needed",
    temperature=0.1,
)

## Load the system prompt from MLflow

Same prompt registered in chapter 2 — keep the hospital helpdesk persona.

In [ ]:
sys_prompt_mlflow = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}@latest")
sys_prompt_plaintext = next(m["content"] for m in sys_prompt_mlflow.format() if m["role"] == "system")
print(sys_prompt_plaintext)

## Build the LangGraph agent

`MultiServerMCPClient` connects to the MCP server and converts its tool schemas into LangChain tools.  
`create_react_agent` builds a ReAct graph: the model node decides which tool to call, the tool node runs it, and the result loops back to the model until it has a final answer — all in **your code**, not on the server.

In [ ]:
async def ask(message: str) -> str:
    """Run one question through the LangGraph agent with MLflow tracing."""
    mcp_client = MultiServerMCPClient(
        {"ticketing": {"transport": "streamable_http", "url": MCP_TICKETING_URL}}
    )
    tools = await mcp_client.get_tools()
    agent = create_react_agent(model, tools, prompt=sys_prompt_plaintext)

    with mlflow.start_span(name=message, span_type="CHAIN") as root_span:
        root_span.set_inputs(message)

        result = await agent.ainvoke(
            {"messages": [("user", message)]},
            config={"recursion_limit": 10},  # ~5 tool calls max
        )

        for msg in result["messages"]:
            # Print tool calls made by the model (with arguments)
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  → {tc['name']}({tc['args']})")
            # Log tool results to MLflow
            if msg.type == "tool":
                with mlflow.start_span(name=f"tool:{msg.name}", span_type="TOOL") as span:
                    span.set_inputs({"tool": msg.name})
                    span.set_outputs(msg.content)

        answer = result["messages"][-1].content
        root_span.set_outputs(answer)

    print(answer)
    return answer

## Try it

These four questions exercise different tools on the server.  
Notice that the intermediate tool calls are now printed in the cell output — because the loop runs here, not on the server.

In [ ]:
messages = [
    # get_ticket
    "Can you check the status of ticket TKT-002 for me?",
    # list_tickets
    "What open tickets do we currently have?",
    # create_ticket
    "A staff member called Mark Davies on Ward 6C says his network connection has been dropping every hour since this morning. Can you raise a ticket for him?",
    # update_ticket
    "Please mark ticket TKT-003 as resolved and add a comment that the paper jam was cleared.",
]

In [ ]:
for message in messages:
    await ask(message)